In [ ]:
from glob import glob
import json
import pandas as pd

files = glob("../experiments/MMSCI/retrival_experiments/*.json")

projects_topics = {
    'biomedical_healthsciences': ["Immunology", "Microbiology", "Diseases", "Pathogenesis", "Oncology"],
    'neroscience_psychology':["Neuroscience", "Neurology", "Psychology", "Anatomy", "Physiology"],
    'genomics_biotechnology':["Biotechnology", "Genetics", "Molecular biology", "Stem cells", "Biochemistry"],
    'environmental_earthscience':["Environmental sciences", "Biogeochemistry", "Ecology", "Solid Earth sciences", "Hydrology"],
    'computationa_datasciences':["Computational biology and bioinformatics", "Mathematical and computing", "Bioinformatics", "Systems biology", "Artificial Intelligence"],
    'chemistry_chemicalsciences':["Chemistry", "Chemical biology", "Materials science", "Biochemistry", "Chemical engineering"],
    'engineering_technologicalinnovation':["Engineering", "Nanoscience and technology", "Optics and photonics", "Energy science and technology", "Materials science"],
    'space_physicalsciences': ["Space physics", "Astronomy and planetary science", "Physics", "Optics and photonics", "Solid Earth sciences"],
    'socialsciences_globaldevelopment':["Social sciences", "Environmental social sciences", "Business and industry", "Developing world", "Risk factors"],
    'agriculture_lifesciences':['Agriculture', 'Plant sciences', 'Zoology', 'Ecology', 'Evolution']

}

dict_topic_to_project = {}
for project in projects_topics.keys():
    for topic in projects_topics[project]:
        if topic not in dict_topic_to_project.keys():
            dict_topic_to_project[topic] = []
        dict_topic_to_project[topic].append(project)

dict_topic_to_project

results = []
for file in files:
    with open(file, "r") as f:
        data = json.load(f)
        result =  data[1:]
        for d in result:
            d['k'] = int(data[0]['k'])
            d['fetch_k'] = int(data[0]['fetch_k'])
            d['threshold'] = float(data[0]['threshold'])
            d['retrieved_loras'] = d['retrieved_loras'].split("-")

            if d['subject'] in dict_topic_to_project.keys():
                d['tag'] = dict_topic_to_project[d['subject']]
                d['loaded_correctly'] = 1 if any(l in d['retrieved_loras'] for l in dict_topic_to_project[d['subject']]) else 0
                d['number_of_retrieved'] = len(d['retrieved_loras'])
                for tag in d['tag']:
                    d['lora'] = tag
                    results.append(d)

            else:
              continue

data = pd.DataFrame(results)


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(font_scale=1.6)

x_ticks = list(set(data['k']))
sns.set_style(style='ticks')
f = sns.relplot(x='k', y='loaded_correctly', hue='lora', style='lora',kind="line", data=data, markers=True, row="fetch_k", col="threshold", legend=True,)
sns.move_legend(f, loc='lower center', ncol=6, bbox_to_anchor=(.45, 1), title='MMSci LoRAs')

plt.xticks(x_ticks)

plt.savefig("../figs/task_mmsci_big_task.pdf", bbox_inches = 'tight',
    pad_inches = 0)


In [ ]:
sns.set_theme(font_scale=1.6)

x_ticks = list(set(data['k']))
sns.set_style(style='ticks')
f = sns.relplot(x='k', y='number_of_retrieved', hue='category', style='category',kind="line", data=data, markers=True, row="fetch_k", col="threshold", legend=True,)
sns.move_legend(f, loc='lower center', ncol=6, bbox_to_anchor=(.45, 1), title='MMSci LoRAs')

plt.xticks(x_ticks)

plt.savefig("../figs/task_mmsci_big_num_loras.pdf", bbox_inches = 'tight',
    pad_inches = 0)


In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

data_filtered = data[data['fetch_k'] == 10]
data_filtered = data_filtered[data_filtered['threshold'] == 0.0]


sns.set_style(style='ticks')
fig, ax = plt.subplots(1,2,  figsize=(15, 6.1))

ax[0] = sns.lineplot(x='k', y='loaded_correctly', hue='lora', style='lora', data=data_filtered, markers=True, ax=ax[0], legend=None)
ax[1] = sns.lineplot(x='k', y='number_of_retrieved', hue='lora', style='lora', data=data_filtered, markers=True, ax=ax[1])

sns.move_legend(ax[1], "upper left", bbox_to_anchor=(1, 1.02),title='MMSCi Dataset', frameon=False)
ax[0].set(ylabel='% of Correct Retrieved LoRAs')
ax[1].set(ylabel='#Retrieved LoRAs')

ax[0].set_xticks(x_ticks)
ax[1].set_xticks(x_ticks)


ax[0].set(xlabel='k\n(a)')
ax[1].set(xlabel='k\n(b)')

ax[0].grid()
ax[1].grid()
plt.tight_layout()
plt.savefig("../figs/retrieved_domains_mmsci_combined.pdf", bbox_inches='tight')



In [ ]:
import json
import pandas as pd
with open("../experiments/MMSCI/inference/d40eac9598316e0d6d327e1debffb63f.json", "r") as fp:
    data = json.load(fp)

full_data = []
for d in data[1:]:
    d['correct'] = 1 if (d['answer'] in d['prediction'][0].split(":")[0]) else 0
    d['num_retrieved'] = len(d['retrieved_loras'])
    full_data.append(d)

full_data=pd.DataFrame(full_data)

perc_correct = full_data['correct'].sum()/len(full_data)
perc_correct